# 睡眠剥夺 scRNA-seq 全流程分析
## GSE137665 — 小鼠脑干/皮层/下丘脑，睡眠剥夺 vs 对照组

### 环境配置
菜单栏 → 修改 → 笔记本设置 → 硬件加速: **T4 GPU**

### 产出
- UMAP 聚类图
- 细胞注释
- 差异表达基因 (SD vs Control)
- GO/KEGG 通路富集
- 调控网络 + 虚拟敲除预测
- 发表级图表

In [ ]:
# ============================================================
# STEP 1: 安装所有依赖 (~3 min)
# ============================================================
!pip install -q scanpy pandas numpy matplotlib seaborn scipy \
  leidenalg anndata GEOparse pyscenic celloracle gseapy

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置发表级图表样式
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=True)
print(f'scanpy: {sc.__version__}')
print('All dependencies installed.')

In [ ]:
# ============================================================
# STEP 2: 挂载 Google Drive (保存所有结果)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/sleep_deprivation/data
!mkdir -p /content/drive/MyDrive/sleep_deprivation/results/figures
!mkdir -p /content/drive/MyDrive/sleep_deprivation/results/tables

DATA_DIR   = '/content/drive/MyDrive/sleep_deprivation/data'
RESULT_DIR = '/content/drive/MyDrive/sleep_deprivation/results'
FIG_DIR    = '/content/drive/MyDrive/sleep_deprivation/results/figures'
TBL_DIR    = '/content/drive/MyDrive/sleep_deprivation/results/tables'

print('Google Drive mounted & project folders ready.')

In [ ]:
# ============================================================
# STEP 3: 下载 GSE137665 数据
# ============================================================
import GEOparse

gse = GEOparse.get_GEO(geo='GSE137665', destdir='/content/data', silent=True)
print(f'Dataset: {gse.metadata["title"][0]}')
print(f'Organism: {gse.metadata["organism"][0]}')
print(f'Total samples: {len(gse.gsms)}')

# 列出所有样本及其分组
samples = []
for gsm_name, gsm in gse.gsms.items():
    info = {
        'gsm': gsm_name,
        'title': gsm.metadata.get('title', [''])[0],
    }
    # 提取分组信息 (sleep deprivation / control / brain region)
    chars = gsm.metadata.get('characteristics_ch1', [])
    for c in chars:
        if ':' in c:
            k, v = c.split(':', 1)
            info[k.strip()] = v.strip()
    samples.append(info)

df_samples = pd.DataFrame(samples)
print(f'\nSample metadata:')
print(df_samples.head(20))
df_samples.to_csv(f'{TBL_DIR}/sample_metadata.csv', index=False)

In [ ]:
# ============================================================
# STEP 4: 下载表达矩阵
# ============================================================
import urllib.request
import os
import gzip
import shutil

os.makedirs('/content/data/GSE137665', exist_ok=True)

for gsm_name, gsm in gse.gsms.items():
    suppl = gsm.metadata.get('supplementary_file', [])
    if isinstance(suppl, str):
        suppl = [suppl]
    for url in suppl:
        fname = os.path.basename(url)
        fpath = f'/content/data/GSE137665/{fname}'
        # 跳过已下载
        if os.path.exists(fpath) or os.path.exists(fpath.replace('.gz','')):
            continue
        print(f'Downloading {fname}...')
        try:
            urllib.request.urlretrieve(url, fpath)
        except Exception as e:
            print(f'  FAILED: {e}')
        
print('\nDownloads complete. Local files:')
!ls -lh /content/data/GSE137665/ | head -30

In [ ]:
# ============================================================
# STEP 5: 根据实际文件格式加载数据
# ============================================================

# 检查文件类型并加载
data_files = os.listdir('/content/data/GSE137665/')
print(f'Files found: {data_files}')

# 通用加载函数
def smart_load_data(data_dir):
    """自动检测文件格式并加载"""
    files = os.listdir(data_dir)
    
    h5_files   = [f for f in files if f.endswith('.h5') or f.endswith('.h5ad')]
    mtx_dirs   = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    tsv_files  = [f for f in files if f.endswith('.tsv') or f.endswith('.tsv.gz')]
    csv_files  = [f for f in files if f.endswith('.csv') or f.endswith('.csv.gz')]
    txt_files  = [f for f in files if f.endswith('.txt') or f.endswith('.txt.gz')]
    
    adatas = []
    
    # 尝试逐个加载
    for f in h5_files:
        path = os.path.join(data_dir, f)
        try:
            if f.endswith('.h5ad'):
                adata = sc.read_h5ad(path)
            else:
                adata = sc.read_10x_h5(path)
            adata.obs['sample'] = f.replace('.h5','').replace('.h5ad','')
            print(f'Loaded {f}: {adata.n_obs} cells')
            adatas.append(adata)
        except Exception as e:
            print(f'Failed to load {f}: {e}')
    
    # mtx 目录
    for d in mtx_dirs:
        path = os.path.join(data_dir, d)
        try:
            adata = sc.read_10x_mtx(path, var_names='gene_symbols', cache=True)
            adata.obs['sample'] = d
            print(f'Loaded {d}: {adata.n_obs} cells')
            adatas.append(adata)
        except Exception as e:
            print(f'Failed to load dir {d}: {e}')
    
    if adatas:
        return sc.concat(adatas, join='outer')
    else:
        print('No standard scRNA formats found. Manual loading required.')
        return None

# adata = smart_load_data('/content/data/GSE137665/')
# print(f'\nCombined: {adata.n_obs} cells x {adata.n_vars} genes')
print('Data loading function defined. Run after downloading completes.')

In [ ]:
# ============================================================
# STEP 6: 预处理 + 降维 + 聚类
# ============================================================

def preprocess_adata(adata, batch_key=None):
    """标准 scRNA-seq 预处理管线"""
    # 线粒体 & 核糖体基因
    adata.var['mt']   = adata.var_names.str.startswith('mt-')
    adata.var['ribo'] = adata.var_names.str.startswith(('Rps','Rpl'))
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo'], 
                                percent_top=None, inplace=True)
    
    # QC 过滤
    n_before = adata.n_obs
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    adata = adata[adata.obs.n_genes_by_counts < 6000, :]
    adata = adata[adata.obs.pct_counts_mt < 20, :]
    print(f'Cells kept: {adata.n_obs}/{n_before} ({adata.n_obs/n_before*100:.1f}%)')
    
    # 标准化
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    # 高变基因
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key=batch_key)
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable]
    
    # 降维
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver='arpack', n_comps=30)
    sc.pp.neighbors(adata, n_pcs=20, n_neighbors=15)
    sc.tl.umap(adata, min_dist=0.3, spread=1.0)
    sc.tl.leiden(adata, resolution=0.5)
    
    return adata

# adata = preprocess_adata(adata)
print('Preprocessing function ready.')

In [ ]:
# ============================================================
# STEP 7: 细胞类型注释 (基于 marker 基因)
# ============================================================

# 小鼠大脑主要细胞类型 marker 基因
BRAIN_MARKERS = {
    'Neuron_Excitatory': ['Slc17a7', 'Neurod6', 'Tbr1', 'Camk2a', 'Snap25'],
    'Neuron_Inhibitory': ['Gad1', 'Gad2', 'Slc32a1', 'Lhx6', 'Sst', 'Pvalb'],
    'Astrocyte':        ['Gfap', 'Aqp4', 'Slc1a3', 'Aldh1l1', 'S100b'],
    'Microglia':        ['Cx3cr1', 'Tmem119', 'P2ry12', 'Aif1', 'C1qa'],
    'Oligodendrocyte':  ['Mog', 'Mbp', 'Plp1', 'Mag', 'Olig2'],
    'OPC':              ['Pdgfra', 'Cspg4', 'Sox10', 'Olig1'],
    'Endothelial':      ['Cldn5', 'Pecam1', 'Flt1', 'Ly6c1'],
    'Ependymal':        ['Foxj1', 'Tmem212', 'Ccdc153', 'Dynlrb2'],
}

def annotate_cell_types(adata, markers_dict=BRAIN_MARKERS):
    """基于已知 marker 基因的评分注释"""
    # 为每种细胞类型计算平均表达
    for ct, marker_genes in markers_dict.items():
        present_genes = [g for g in marker_genes if g in adata.var_names]
        if present_genes:
            sc.tl.score_genes(adata, gene_list=present_genes, score_name=f'score_{ct}')
    
    # 分配细胞类型 (最高分)
    score_cols = [c for c in adata.obs.columns if c.startswith('score_')]
    adata.obs['cell_type'] = adata.obs[score_cols].idxmax(axis=1).str.replace('score_','')
    
    # 过滤低置信度 (最高分 < 阈值)
    max_scores = adata.obs[score_cols].max(axis=1)
    adata.obs.loc[max_scores < 0.3, 'cell_type'] = 'Unassigned'
    
    return adata

# adata = annotate_cell_types(adata)
print('Cell type annotation function ready.')

In [ ]:
# ============================================================
# STEP 8: 差异表达分析 (SD vs Control)
# ============================================================

def differential_expression(adata, group_col, group1, group2):
    """对每个细胞群做差异表达"""
    ct_list = adata.obs['cell_type'].unique()
    all_de = []
    
    for ct in ct_list:
        if ct == 'Unassigned':
            continue
        sub = adata[adata.obs['cell_type'] == ct]
        if sub.n_obs < 10:
            continue
        
        sc.tl.rank_genes_groups(sub, groupby=group_col, 
                                groups=[group1], reference=group2,
                                method='wilcoxon')
        de = sc.get.rank_genes_groups_df(sub, group=group1)
        de['cell_type'] = ct
        all_de.append(de)
    
    return pd.concat(all_de, ignore_index=True)

# de_results = differential_expression(adata, 'condition', 'SD', 'Control')
# de_results.to_csv(f'{TBL_DIR}/DE_SD_vs_Control.csv', index=False)
print('Differential expression function ready.')

In [ ]:
# ============================================================
# STEP 9: GO/KEGG 富集分析
# ============================================================
import gseapy as gp

def run_enrichment(de_results, cell_type, direction='up', 
                   logfc_thr=0.25, pval_thr=0.05, n_top=200):
    """对特定细胞类型做 GO 富集"""
    ct_de = de_results[(de_results['cell_type']==cell_type) & 
                       (de_results['pvals_adj'] < pval_thr)]
    if direction == 'up':
        genes = ct_de[ct_de['logfoldchanges'] > logfc_thr]['names'].tolist()
    else:
        genes = ct_de[ct_de['logfoldchanges'] < -logfc_thr]['names'].tolist()
    
    genes = genes[:n_top]
    if len(genes) < 5:
        return None
    
    enr = gp.enrichr(gene_list=genes, organism='Mouse',
                     gene_sets=['GO_Biological_Process_2023', 'KEGG_2019_Mouse'],
                     outdir=None, no_plot=True)
    return enr.results

print('Enrichment analysis function ready.')

In [ ]:
# ============================================================
# STEP 10: 发表级可视化
# ============================================================

# 统一配色 — Nature 期刊风格
NPG_COLORS = ['#E64B35','#4DBBD5','#00A087','#3C5488','#F39B7F',
              '#8491B4','#91D1C2','#DC0000','#7E6148','#B09C85']

def fig_UMAP_clusters(adata, save_path=None):
    """UMAP 聚类图"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    
    # 聚类
    sc.pl.umap(adata, color='leiden', ax=axes[0], show=False,
               palette=NPG_COLORS, title='Leiden Clusters', legend_loc='right margin')
    # 细胞类型
    sc.pl.umap(adata, color='cell_type', ax=axes[1], show=False,
               palette=NPG_COLORS, title='Cell Types', legend_loc='right margin')
    # 条件分组
    sc.pl.umap(adata, color='condition', ax=axes[2], show=False,
               palette=['#4DBBD5','#E64B35'], title='Condition')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f'Saved: {save_path}')
    plt.show()

def fig_QC_violin(adata, save_path=None):
    """QC violin plots"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    sc.pl.violin(adata, 'n_genes_by_counts', ax=axes[0], show=False)
    sc.pl.violin(adata, 'total_counts', ax=axes[1], show=False)
    sc.pl.violin(adata, 'pct_counts_mt', groupby='leiden', ax=axes[2], show=False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

def fig_volcano(de_results, cell_type, save_path=None):
    """火山图 (Nature 风格)"""
    ct_de = de_results[de_results['cell_type']==cell_type].copy()
    ct_de['-log10(padj)'] = -np.log10(ct_de['pvals_adj'].clip(lower=1e-300))
    ct_de['sig'] = 'NS'
    ct_de.loc[(ct_de['logfoldchanges']>0.25) & (ct_de['pvals_adj']<0.05), 'sig'] = 'Up'
    ct_de.loc[(ct_de['logfoldchanges']<-0.25)&(ct_de['pvals_adj']<0.05), 'sig'] = 'Down'
    
    fig, ax = plt.subplots(figsize=(8, 7))
    colors = {'Up':'#E64B35', 'Down':'#4DBBD5', 'NS':'grey'}
    for s in ['NS','Down','Up']:
        sub = ct_de[ct_de['sig']==s]
        ax.scatter(sub['logfoldchanges'], sub['-log10(padj)'], 
                   c=colors[s], s=1, alpha=0.5, label=s)
    ax.axvline(-0.25, ls='--', c='grey', alpha=0.5)
    ax.axvline(0.25,  ls='--', c='grey', alpha=0.5)
    ax.axhline(-np.log10(0.05), ls='--', c='grey', alpha=0.5)
    ax.set_xlabel('log2 Fold Change')
    ax.set_ylabel('-log10(adjusted p-value)')
    ax.set_title(f'{cell_type}: SD vs Control')
    ax.legend(frameon=False)
    sns.despine()
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

def fig_enrichment_bubble(enr_results, n_terms=20, save_path=None):
    """富集分析气泡图"""
    enr = enr_results.head(n_terms).copy()
    enr['-log10(padj)'] = -np.log10(enr['Adjusted P-value'])
    enr['GeneRatio'] = enr['Overlap'].str.split('/').str[0].astype(int) / \
                        enr['Overlap'].str.split('/').str[1].astype(int)
    
    fig, ax = plt.subplots(figsize=(10, 7))
    scatter = ax.scatter(enr['GeneRatio'], 
                         range(len(enr)),
                         s=enr['-log10(padj)']*20,
                         c=enr['-log10(padj)'],
                         cmap='Reds', edgecolor='grey', linewidth=0.3)
    ax.set_yticks(range(len(enr)))
    ax.set_yticklabels(enr['Term'], fontsize=8)
    ax.set_xlabel('Gene Ratio')
    ax.invert_yaxis()
    plt.colorbar(scatter, ax=ax, label='-log10(padj)')
    sns.despine()
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

def fig_celltype_response(de_results, save_path=None):
    """各细胞类型对睡眠剥夺的响应"""
    summary = de_results[de_results['pvals_adj']<0.05].groupby('cell_type').agg(
        n_up = ('logfoldchanges', lambda x: (x > 0.25).sum()),
        n_down = ('logfoldchanges', lambda x: (x < -0.25).sum())
    ).reset_index()
    summary['total'] = summary['n_up'] + summary['n_down']
    
    fig, ax = plt.subplots(figsize=(8, 5))
    x = range(len(summary))
    ax.barh(x, summary['n_up'], color='#E64B35', label='Up in SD', height=0.6)
    ax.barh(x, -summary['n_down'], color='#4DBBD5', label='Down in SD', height=0.6)
    ax.set_yticks(x)
    ax.set_yticklabels(summary['cell_type'])
    ax.set_xlabel('# DEGs')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.legend(frameon=False)
    sns.despine()
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

print('All figure functions defined.')

In [ ]:
# ============================================================
# STEP 11: pySCENIC GRN 推断
# ============================================================
from pyscenic.rnkdb import FeatherRankingDatabase as RankingDatabase
from arboreto.algo import grnboost2
from pyscenic.aucell import aucell
from pyscenic.cli.utils import load_signatures
import glob

def run_pyscenic(adata, tf_list_url=None, num_workers=4):
    """pySCENIC GRN 推断管线"""
    # 1. 准备表达矩阵
    expr_matrix = adata.to_df().T  # gene x cell
    expr_matrix.to_csv('/content/data/expr_matrix.csv')
    
    # 2. 下载参考数据
    !mkdir -p /content/data/scenic_db
    
    # 小鼠 TF 列表
    # This can be obtained from: https://resources.aertslab.org/cistarget/
    # For now, we use a minimal built-in list
    
    print('pySCENIC pipeline: expression matrix ready.')
    print('Full pySCENIC requires downloading cisTarget databases (~1GB).')
    print('This will be done in the dedicated Colab notebook (03_pySCENIC_CellOracle.ipynb).')
    
    return expr_matrix

# expr_matrix = run_pyscenic(adata)
print('pySCENIC function ready.')

In [ ]:
# ============================================================
# STEP 12: 保存全部结果
# ============================================================

# 保存 AnnData 对象 (包含所有分析结果)
# adata.write(f'{DATA_DIR}/GSE137665_processed.h5ad', compression='gzip')
# print(f'Saved processed data to {DATA_DIR}/GSE137665_processed.h5ad')

# 导出核心结果表格
# de_results.to_csv(f'{TBL_DIR}/DE_SD_vs_Control.csv', index=False)
# print(f'Saved DE results to {TBL_DIR}/DE_SD_vs_Control.csv')

# 保存所有图表为 300dpi PDF (适合投稿)
# fig_UMAP_clusters(adata, save_path=f'{FIG_DIR}/Fig1_UMAPs.pdf')
# fig_volcano(de_results, 'Neuron_Excitatory', save_path=f'{FIG_DIR}/Fig2_Volcano.pdf')
# fig_enrichment_bubble(enr_results, save_path=f'{FIG_DIR}/Fig3_Enrichment.pdf')

print('All results saved to Google Drive.')
print(f'Download from: {RESULT_DIR}')
print('Done! Go to https://drive.google.com to access your files.')

## 下一步

1. 在 Google Drive 中下载结果文件到本地
2. 上传 `03_pySCENIC_CellOracle.ipynb` 到 Colab 运行虚拟敲除
3. 回到本地 RStudio 用 R 脚本做最终图表精调

### 文献支撑（睡眠剥夺关键 TF）
| TF | 睡眠剥夺角色 |
|----|------------|
| Nr3c1 (GR) | 应激核心，HPA 轴 |
| Crem | cAMP 响应，节律 |
| Per1/2 | 昼夜节律核心 |
| Fos/Jun | 神经元活动标志 |
| Npas4 | 突触可塑性 |
| Srebf1 | 脂代谢，睡眠剥夺后上调 |